# bitsandbytes smoke test

k-bit quantization library (LLM.int8(), NF4/FP4, 8-bit optimizers). The compiled
CUDA backend (`libbitsandbytes_cuda<ver>.so`) drives GPU kernels; the pure-torch
"default"/CPU backend covers blockwise-8bit and 4-bit quant on CPU. Cells that
need real GPU kernels (int8 matmul, Linear4bit weight-quant, 8-bit optimizer
step) are tagged `requires-gpu`: they run only with `--with gpu` and are skipped
otherwise.

In [ ]:
# Blockwise 8-bit quantization — bitsandbytes' core int8 primitive. Runs on the
# CPU/"default" backend (no GPU needed). Round-trip and assert bounded error.
import torch
import bitsandbytes.functional as F
torch.manual_seed(0)

x = torch.randn(512, 512, dtype=torch.float32)
q, state = F.quantize_blockwise(x)            # -> uint8 codes + QuantState
x_rec = F.dequantize_blockwise(q, state)

assert q.dtype == torch.uint8, q.dtype
assert x_rec.shape == x.shape
err = (x - x_rec).abs().mean().item()
assert err < 0.05, f"blockwise reconstruction error too high: {err}"
print(f"blockwise int8: codes={tuple(q.shape)} dtype={q.dtype} mean_abs_err={err:.4f}")

In [ ]:
# 4-bit quantization (QLoRA's NF4 / FP4). quantize_4bit packs two 4-bit codes per
# uint8 byte; dequantize_4bit restores. CPU-capable. NF4 (info-theoretically
# tuned for normal weights) should reconstruct at least as well as FP4.
import torch
import bitsandbytes.functional as F
torch.manual_seed(0)

w = torch.randn(1024, 256, dtype=torch.float32)   # normally-distributed "weights"
errs = {}
for qt in ("nf4", "fp4"):
    q, state = F.quantize_4bit(w, quant_type=qt, blocksize=64)
    w_rec = F.dequantize_4bit(q, state)
    assert q.dtype == torch.uint8 and q.numel() * 2 == w.numel()  # packed 2-per-byte
    errs[qt] = (w - w_rec).abs().mean().item()

assert errs["nf4"] <= errs["fp4"] + 1e-3, errs
print(f"4bit mean_abs_err: nf4={errs['nf4']:.4f}  fp4={errs['fp4']:.4f}  (nf4 <= fp4)")

In [ ]:
# Double (nested) quantization — QLoRA's memory trick: the per-block absmax
# scales are THEMSELVES 8-bit quantized into a second QuantState (state.state2).
# CPU-capable. Verify the nested state exists and the round-trip still holds.
import torch
import bitsandbytes.functional as F
torch.manual_seed(0)

w = torch.randn(2048, 256, dtype=torch.float32)
q, state = F.quantize_4bit(w, quant_type="nf4", compress_statistics=True)
assert state.state2 is not None, "compress_statistics did not produce a nested state2"
w_rec = F.dequantize_4bit(q, state)
err = (w - w_rec).abs().mean().item()
assert w_rec.shape == w.shape and err < 0.12, (tuple(w_rec.shape), err)
print(f"double-quant nf4: nested absmax present, mean_abs_err={err:.4f}")

In [ ]:
# Layers that run on the CPU backend: an LLM.int8() linear layer's forward pass,
# Params4bit construction (it stays unquantized until it is moved to the GPU) and
# StableEmbedding, the drop-in replacement for nn.Embedding.
import torch, bitsandbytes as bnb
torch.manual_seed(0)

layer = bnb.nn.Linear8bitLt(64, 32, bias=True, has_fp16_weights=False)
x = torch.randn(8, 64, dtype=torch.float32)
y = layer(x)
assert y.shape == (8, 32) and torch.isfinite(y).all()
print(f"Linear8bitLt CPU forward: {tuple(x.shape)} -> {tuple(y.shape)} dtype={y.dtype}")

p = bnb.nn.Params4bit(torch.randn(32, 16), quant_type="nf4")   # unquantized until .cuda()
assert p.quant_type == "nf4" and tuple(p.data.shape) == (32, 16)
emb = bnb.nn.StableEmbedding(16, 8)
assert emb(torch.tensor([0, 3, 5])).shape == (3, 8)
print("CPU: Params4bit + StableEmbedding constructed")

In [ ]:
# LLM.int8() matmul on the GPU: fp16 activations through Linear8bitLt's int8 kernel.
import torch, bitsandbytes as bnb
torch.manual_seed(0)

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

x = torch.randn(8, 64, dtype=torch.float32)
g_layer = bnb.nn.Linear8bitLt(64, 32, bias=True, has_fp16_weights=False).cuda()
g_y = g_layer(x.cuda().to(torch.float16))            # int8 matmul: fp16 activations
assert g_y.shape == (8, 32) and torch.isfinite(g_y).all()
print(f"Linear8bitLt GPU int8-matmul kernel -> {tuple(g_y.shape)} dtype={g_y.dtype}")

In [ ]:
# Linear4bit + Adam8bit quantize their state on the .cuda() move, so the module
# forward and the optimizer step are GPU-only paths.
import torch, bitsandbytes as bnb
torch.manual_seed(0)

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

lin = bnb.nn.Linear4bit(16, 8, quant_type="nf4", compute_dtype=torch.float16).cuda()
y = lin(torch.randn(4, 16, device="cuda", dtype=torch.float16))
assert y.shape == (4, 8) and torch.isfinite(y).all()
w = torch.nn.Parameter(torch.randn(8, 8, device="cuda"))
opt = bnb.optim.Adam8bit([w], lr=1e-3)
(w ** 2).sum().backward(); opt.step()
print(f"GPU: Linear4bit forward -> {tuple(y.shape)}; Adam8bit step OK")